In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import os
import warnings

warnings.filterwarnings('ignore') # Gereksiz syntax uyarılarını gizler

data_dir = "../data"
files = {
    "BIST100": "bist100_weekly.csv",
    "VIX": "vix_weekly.csv",
    "GOLD_TL": "gold_tl_weekly.csv",
    "BOND_2Y": "bond_2y_weekly.csv"
}

def load_and_clean(file_name, col_name):
    path = os.path.join(data_dir, file_name)
    
    df = pd.read_csv(path, sep=None, engine='python')
    df = df.iloc[:, [0, 1]].copy()
    df.columns = ["Date", col_name]
    
    print(f"--- {col_name} ---")
    print(f"1. Başlangıç Satır Sayısı: {len(df)}")
    print(f"2. Örnek Tarih: '{df['Date'].iloc[0]}' | Örnek Fiyat: '{df[col_name].iloc[0]}'")
    
    # GARANTİ TEMİZLİK: Ne tür veri gelirse gelsin zorla sayıya (float) çevirir
    df[col_name] = df[col_name].astype(str)
    df[col_name] = df[col_name].str.replace(r'\.', '', regex=True) # Syntax warning çözüldü
    df[col_name] = df[col_name].str.replace(',', '.', regex=False)
    df[col_name] = df[col_name].str.extract(r'([-+]?\d*\.?\d+)')[0] # Sadece rakamları çeker
    df[col_name] = pd.to_numeric(df[col_name], errors='coerce')
    
    aylar = {"Oca": "Jan", "Şub": "Feb", "Mar": "Mar", "Nis": "Apr", "May": "May", "Haz": "Jun",
             "Tem": "Jul", "Ağu": "Aug", "Eyl": "Sep", "Eki": "Oct", "Kas": "Nov", "Ara": "Dec"}
    for tr, en in aylar.items():
        df["Date"] = df["Date"].astype(str).str.replace(tr, en, regex=False)
        
    df["Date"] = pd.to_datetime(df["Date"], format='mixed', dayfirst=True, errors='coerce')
    
    df = df.dropna(subset=["Date", col_name])
    df = df.set_index("Date").sort_index()
    print(f"3. Temizlik Sonrası Satır Sayısı: {len(df)}\n")
    return df

dfs = []
for key, file in files.items():
    dfs.append(load_and_clean(file, key))

df_merged = pd.concat(dfs, axis=1, sort=True)
print(f"-> Tüm veriler birleşti. Toplam Satır: {len(df_merged)}")

df_merged = df_merged.resample('W-FRI').last().ffill().bfill()
print(f"-> Haftalık Cuma hizalaması yapıldı. Toplam Satır: {len(df_merged)}")

# Logaritma çökmesini engellemek için tablonun tamamı FLOAT olarak zorlanıyor
df_merged = df_merged.astype(float)

df_log_diff = np.log(df_merged).diff().dropna()
print(f"-> Logaritma ve Fark alındıktan sonra kalan Satır: {len(df_log_diff)}\n")

if len(df_log_diff) == 0:
    raise ValueError("HATA: Veri seti boş!")

scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(df_log_diff)
df_scaled = pd.DataFrame(scaled_data, columns=df_log_diff.columns, index=df_log_diff.index)

df_scaled = df_scaled[['GOLD_TL', 'VIX', 'BOND_2Y', 'BIST100']]

if len(df_scaled) >= 720:
    df_scaled = df_scaled.tail(720) 
    train_df = df_scaled.iloc[:540]
    test_df = df_scaled.iloc[540:720]
else:
    print(f"UYARI: Elimizde sadece {len(df_scaled)} hafta var. 720 eksik, oranlayarak bölüyoruz.")
    split_idx = int(len(df_scaled) * 0.75)
    train_df = df_scaled.iloc[:split_idx]
    test_df = df_scaled.iloc[split_idx:]

train_df.to_csv(os.path.join(data_dir, "processed_train.csv"))
test_df.to_csv(os.path.join(data_dir, "processed_test.csv"))

print(f"✅ VERİ İŞLEME BAŞARIYLA TAMAMLANDI!")
print(f"Eğitim Seti (Train) Boyutu: {train_df.shape}")
print(f"Test Seti Boyutu: {test_df.shape}")

--- BIST100 ---
1. Başlangıç Satır Sayısı: 731
2. Örnek Tarih: '31.12.2023' | Örnek Fiyat: '7.628,73'
3. Temizlik Sonrası Satır Sayısı: 731

--- VIX ---
1. Başlangıç Satır Sayısı: 52
2. Örnek Tarih: '08.11.2026' | Örnek Fiyat: '19,30'
3. Temizlik Sonrası Satır Sayısı: 52

--- GOLD_TL ---
1. Başlangıç Satır Sayısı: 3651
2. Örnek Tarih: '29.12.2023' | Örnek Fiyat: '60.798'
3. Temizlik Sonrası Satır Sayısı: 3651

--- BOND_2Y ---
1. Başlangıç Satır Sayısı: 731
2. Örnek Tarih: '31.12.2023' | Örnek Fiyat: '37,880'
3. Temizlik Sonrası Satır Sayısı: 731

-> Tüm veriler birleşti. Toplam Satır: 4434
-> Haftalık Cuma hizalaması yapıldı. Toplam Satır: 881
-> Logaritma ve Fark alındıktan sonra kalan Satır: 880

✅ VERİ İŞLEME BAŞARIYLA TAMAMLANDI!
Eğitim Seti (Train) Boyutu: (540, 4)
Test Seti Boyutu: (180, 4)
